> 在线讲义：[第 16 章 · 硬件组件与控制器](https://wikieden.github.io/ros2-from-scratch/lessons/16-hardware-controllers.html) · 本书《从零学 ROS 2》

# 第 16 章 · 硬件组件与控制器

本章回答：要让 ros2_control 驱动自己的电机，硬件组件要实现哪些函数？上电、松刹车该写在哪？最常用的标准控制器（关节轨迹、差速底盘、关节状态广播）怎么用？自己写控制器要遵守什么？

配套讲义：仓库里 `ros2/notes/lessons/16-hardware-controllers.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 ROS 2。

**为什么是模拟**：硬件组件和控制器都是 C++ 插件。这里用 Python 写出同样的调用顺序；JTC 只做线性插值（真 JTC 用样条，能接速度、加速度），差速公式见讲义附录"代码对照"一节的说明。

In [ ]:
import itertools

print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 规则 / 数字 | 写法 | 跑出来 |
|---|---|---|---|
| 第 1 步 生命周期 | on_init → on_configure → on_activate | `FakeHardware()` | ACTIVE 之前 read 不干活 |
| 第 2 步 JTC 插值 | 路点之间按时间插值 | `interpolate(waypoints, t)` | t=0.5 → 0.5；t=1.5 → 0.75；t=2.5 → 0.5 |
| 同上 | 1 kHz 每拍算一个点 | `[interpolate(wp, k/1000) for k in range(2001)]` | 2001 个点 |
| 第 2 步 差速底盘 | 按轮距、轮径算左右轮速度 | `diff_drive(0.2, 1.0, 0.3, 0.05)` | 左 1.0 rad/s，右 7.0 rad/s |

## 1. 先把要用的东西定义好

这几格只是定义（常量、函数、类），运行后没有输出也正常。用到的前面章节的函数也一起抄进来了，不需要本地文件。

**类 `FakeHardware`**：一个 SystemInterface 的样子：生命周期回调里做的事按讲义第 1 步的表来，只有 ACTIVE 时 read / write 才干活。

In [ ]:
class FakeHardware:
    """一个 SystemInterface 的样子：生命周期回调里做的事按讲义第 1 步的表来，只有 ACTIVE 时 read / write 才干活。"""

    def __init__(self):
        self.state = "UNCONFIGURED"
        self.log: list[str] = []

    def on_init(self):
        self.log.append("on_init：读 URDF 里的参数，检查关节和接口（不和硬件通信）")

    def on_configure(self):
        self.log.append("on_configure：建立通信")
        self.state = "INACTIVE"

    def on_activate(self):
        self.log.append("on_activate：上电、松刹车")
        self.state = "ACTIVE"

    def read(self) -> bool:
        return self.state == "ACTIVE"

    def write(self) -> bool:
        return self.state == "ACTIVE"

**函数 `diff_drive`**：cmd_vel（线速度 v、角速度 ω）→ 左右轮角速度：轮子线速度 = v ∓ ω·轮距/2，再除以轮半径。

In [ ]:
def diff_drive(v: float, w: float, wheel_separation: float, wheel_radius: float) -> tuple[float, float]:
    """cmd_vel（线速度 v、角速度 ω）→ 左右轮角速度：轮子线速度 = v ∓ ω·轮距/2，再除以轮半径。"""
    left = (v - w * wheel_separation / 2) / wheel_radius
    right = (v + w * wheel_separation / 2) / wheel_radius
    return left, right

**函数 `interpolate`**：路点 (到达时刻, 关节位置)；两个路点之间按时间线性插值，过了最后一个就停在终点。

In [ ]:
def interpolate(waypoints: list[tuple[float, float]], t: float) -> float:
    """路点 (到达时刻, 关节位置)；两个路点之间按时间线性插值，过了最后一个就停在终点。"""
    if t <= waypoints[0][0]:
        return waypoints[0][1]
    for (t0, q0), (t1, q1) in itertools.pairwise(waypoints):
        if t0 <= t <= t1:
            return q0 + (q1 - q0) * (t - t0) / (t1 - t0)
    return waypoints[-1][1]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 · 硬件组件：生命周期一步一步走，ACTIVE 之前 read / write 不干活")
hw = FakeHardware()
hw.on_init()
print("  on_init 之后:", hw.state, "；read 干活吗？", hw.read())
hw.on_configure()
hw.on_activate()
print("  activate 之后:", hw.state, "；read 干活吗？", hw.read())
for line in hw.log:
    print("   ", line)

In [ ]:
print("\n第二步 · JTC：路点隔得远，控制器按时间插值，每拍算出当前该到的位置")
waypoints = [(0.0, 0.0), (1.0, 1.0), (2.0, 0.5)]  # 三个路点是编的
for t in [0.0, 0.5, 1.0, 1.5, 2.0, 2.5]:
    print(f"  t={t:.1f} s → joint1 = {interpolate(waypoints, t):.3f}")
samples = [interpolate(waypoints, k / 1000) for k in range(2001)]
print("  1 kHz 下 2 秒一共算了", len(samples), "个点")

In [ ]:
print("\n第二步（续）· 差速底盘：v=0.2 m/s、ω=1.0 rad/s，轮距 0.3 m、轮半径 0.05 m（编的）")
left, right = diff_drive(0.2, 1.0, 0.3, 0.05)
print(f"  左轮 {left:.1f} rad/s，右轮 {right:.1f} rad/s（往左转：右轮快）")

## 3. 代码里才看得到的

1. **差速公式讲义没写，这里补上**：底盘绕中心以角速度 ω 转时，离中心 L/2 的左右轮线速度分别是 $v-\omega L/2$ 和 $v+\omega L/2$（L 是轮距），再除以轮半径 r 得到轮子角速度。数字（轮距 0.3 m、轮半径 0.05 m）是编的。
2. **真 JTC 用样条，不是折线**：只给位置时，线性插值在路点处速度会突变；JTC 用三次 / 五次样条让速度、加速度连续（讲义第 2 步）。
3. **三个路点是编的**；过了最后一个路点就停在终点，和 JTC 执行完保持位置一致。

## 4. 自己改一改

- 给 `waypoints` 加一个 (3.0, 1.0)，看 t = 2.5 时的位置；
- 把 `diff_drive` 的 ω 改成 0（直走），看左右轮速度是不是一样。

在下面的空格子里写，或者直接改上面的格子。讲义末尾的练习先在纸上做，不在这里给答案。

In [ ]:
# 在这里写你的代码

## 口诀

- **硬件组件四步**：init 读配置，configure 通信，activate 上电松刹车，read / write 每拍收发。
- **接口自动导出**：名字 `关节/类型`，read 里 `set_state`，write 里 `get_command`。
- **JTC**：路点 + 时刻，插值执行；话题无反馈，动作有反馈；新轨迹替换旧的。
- **差速**：收 TwistStamped，发 odom 和 tf。
- **写控制器**：声明要哪些接口；update 要实时；订阅回调和 update 之间用实时缓冲区。